In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import issparse

import scanpy as sc
import anndata


## Mycellia Data

In [35]:
basedir = "/home/data/kaggle_data/myllia/"

In [3]:
df1 = pd.read_csv(basedir+"training_data_means.csv")
print(df1.shape)
print(df1.tail())
print(df1.loc[df1["pert_symbol"] == "VEGFA", "VEGFA"])
print(df1.loc[df1["pert_symbol"] == "non-targeting", "VEGFA"])
print(df1.loc[df1["pert_symbol"] == "ARID1A"])
print(df1["pert_symbol"].to_list())

(81, 5128)
      pert_symbol      A1BG      A1CF     AADAC      AAK1     AARS1      AASS  \
76         TGFBR2  0.537884  0.015554  0.106994  0.455972  0.723697  0.005621   
77          USP22  0.443085  0.044874  0.123302  0.288553  0.675256  0.000000   
78          VEGFA  0.498489  0.009164  0.096523  0.420647  0.753090  0.003990   
79            WAC  0.481894  0.023404  0.085533  0.476920  0.801459  0.000000   
80  non-targeting  0.474062  0.025797  0.093244  0.443709  0.737890  0.002591   

       ABCA1    ABCA12     ABCA5  ...       ZP3      ZPBP    ZRANB3   ZSCAN18  \
76  0.162179  0.024584  0.107136  ...  0.311330  0.002471  0.516634  0.011630   
77  0.087892  0.041430  0.095096  ...  0.362282  0.000000  0.468844  0.010474   
78  0.135827  0.017036  0.085875  ...  0.298464  0.000000  0.535847  0.005435   
79  0.152639  0.027180  0.129683  ...  0.253768  0.002552  0.555070  0.010930   
80  0.137953  0.018650  0.099708  ...  0.283082  0.000991  0.526363  0.009778   

     ZSCAN31   

In [37]:
df2 = pd.read_csv(basedir+"sample_submission.csv")
print(df2.shape)
print(df2.head())

(120, 5128)
  pert_id      A1BG      A1CF    AADAC      AAK1     AARS1      AASS  \
0  pert_1  0.011603  0.000986 -0.00299 -0.000908 -0.006582 -0.000028   
1  pert_2  0.011603  0.000986 -0.00299 -0.000908 -0.006582 -0.000028   
2  pert_3  0.011603  0.000986 -0.00299 -0.000908 -0.006582 -0.000028   
3  pert_4  0.011603  0.000986 -0.00299 -0.000908 -0.006582 -0.000028   
4  pert_5  0.011603  0.000986 -0.00299 -0.000908 -0.006582 -0.000028   

      ABCA1    ABCA12     ABCA5  ...       ZP3      ZPBP    ZRANB3   ZSCAN18  \
0 -0.002154 -0.000425  0.010644  ...  0.027073 -0.000251  0.002195  0.002279   
1 -0.002154 -0.000425  0.010644  ...  0.027073 -0.000251  0.002195  0.002279   
2 -0.002154 -0.000425  0.010644  ...  0.027073 -0.000251  0.002195  0.002279   
3 -0.002154 -0.000425  0.010644  ...  0.027073 -0.000251  0.002195  0.002279   
4 -0.002154 -0.000425  0.010644  ...  0.027073 -0.000251  0.002195  0.002279   

    ZSCAN31    ZSWIM5    ZSWIM6    ZSWIM7     ZWINT       ZYX  
0  0.00728

In [5]:
df3 = pd.read_csv(basedir+"pert_ids_val.csv")
print(df3.shape)
print(df3.head())

(60, 3)
      pert class pert_id
0  SMARCE1   val  pert_1
1     DPF2   val  pert_2
2    MRE11   val  pert_3
3   TCF7L2   val  pert_4
4   HMGXB4   val  pert_5


In [6]:
df4 = pd.read_csv(basedir+"training_data_ground_truth_table.csv")
print(df4.shape)
print(df4.head())

(80, 10256)
  pert_id      A1BG      A1CF     AADAC      AAK1     AARS1      AASS  \
0    ACLY -0.162420 -0.003694  0.007704 -0.065038 -0.022033 -0.002591   
1   ALDOA  0.153383  0.018165  0.001091 -0.001647 -0.079765 -0.002591   
2   APAF1 -0.006948  0.004917 -0.002335  0.026446 -0.005120  0.000229   
3   ARID2  0.043123 -0.007021  0.009316  0.001984 -0.050810  0.007644   
4    BAG1 -0.065040  0.009065  0.036724  0.019140  0.081988  0.000408   

      ABCA1    ABCA12     ABCA5  ...    w_ZPBP  w_ZRANB3  w_ZSCAN18  \
0 -0.019053  0.012141 -0.012702  ...  0.005787  3.673932   0.241709   
1  0.062238 -0.018650 -0.038159  ...  0.010308  2.072907   1.071005   
2 -0.019583  0.007200  0.037674  ...  0.039448  0.039448   0.039448   
3 -0.013434 -0.002272  0.023748  ...  0.015171  0.103470   0.015171   
4 -0.012373 -0.015974  0.031339  ...  0.024883  2.457045   0.024883   

   w_ZSCAN31  w_ZSWIM5  w_ZSWIM6  w_ZSWIM7   w_ZWINT     w_ZYX  baseline_wmae  
0   0.005787  0.023946  0.450036  1.130375

In [7]:
adata = sc.read_h5ad(basedir+"training_cells.h5ad")
print(adata)
# print(adata.X)
print(adata.obs)
print(adata.var)
print(adata.obs["sgrna_symbol"].nunique())

AnnData object with n_obs × n_vars = 17882 × 19226
    obs: 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'sgrna_id', 'sgrna_symbol', 'channel'
    var: 'features'
                       nCount_RNA  nFeature_RNA  percent.mt          sgrna_id  \
AAACCAAAGACGCGAA_ch_1     16395.0          4191    4.306191          INSIG1_3   
AAACCAAAGCAAATGA_ch_1     12000.0          3631    3.133333            FLNA_2   
AAACCAAAGCAGTCTA_ch_1     55050.0          7061    5.193460           EIF3H_2   
AAACCAAAGGGCATAG_ch_1     19586.0          5010    6.494435           DZIP3_4   
AAACCATTCCAATCGA_ch_1     18744.0          4697    4.604140  non-targeting_25   
...                           ...           ...         ...               ...   
GTTGTCCGTCCAGTGA_ch_4     21242.0          4904    4.057998          METTL3_4   
GTTGTCCGTGGGTTTG_ch_4     17462.0          4674    3.928531  non-targeting_25   
GTTGTCTTCCCCAACC_ch_4     10459.0          3350    6.922268           IKBKG_1   
GTTGTCTTCCCTTCTC_ch_4     29

In [8]:
df = adata.to_df()
df.head()
print(df.columns.to_list())

['A1BG', 'A1CF', 'A2M', 'A2ML1', 'A3GALT2', 'A4GALT', 'A4GNT', 'AAAS', 'AACS', 'AADAC', 'AADACL2', 'AADACL3', 'AADACL4', 'AADAT', 'AAGAB', 'AAK1', 'AAMDC', 'AAMP', 'AANAT', 'AAR2', 'AARD', 'AARS1', 'AARS2', 'AARSD1', 'AASDH', 'AASDHPPT', 'AASS', 'AATF', 'AATK', 'ABAT', 'ABCA1', 'ABCA10', 'ABCA12', 'ABCA13', 'ABCA2', 'ABCA3', 'ABCA4', 'ABCA5', 'ABCA6', 'ABCA7', 'ABCA8', 'ABCA9', 'ABCB1', 'ABCB10', 'ABCB11', 'ABCB4', 'ABCB5', 'ABCB6', 'ABCB7', 'ABCB8', 'ABCB9', 'ABCC1', 'ABCC10', 'ABCC11', 'ABCC12', 'ABCC2', 'ABCC3', 'ABCC4', 'ABCC5', 'ABCC6', 'ABCC8', 'ABCC9', 'ABCD1', 'ABCD2', 'ABCD3', 'ABCD4', 'ABCE1', 'ABCF1', 'ABCF2', 'ABCF2-H2BK1', 'ABCF3', 'ABCG1', 'ABCG2', 'ABCG4', 'ABCG5', 'ABCG8', 'ABHD1', 'ABHD10', 'ABHD11', 'ABHD12', 'ABHD12B', 'ABHD13', 'ABHD14A', 'ABHD14A-ACY1', 'ABHD14B', 'ABHD15', 'ABHD16A', 'ABHD16B', 'ABHD17A', 'ABHD17B', 'ABHD17C', 'ABHD18', 'ABHD2', 'ABHD3', 'ABHD4', 'ABHD5', 'ABHD6', 'ABHD8', 'ABI1', 'ABI2', 'ABI3', 'ABI3BP', 'ABITRAM', 'ABL1', 'ABL2', 'ABLIM1', 'ABL

In [9]:
pert_df = pd.read_csv(basedir+"pert_ids_all.csv")

# x/var in training data
x1 = pert_df.iloc[:60,:].loc[pert_df["pert"].isin(df4["pert_id"]), "pert"].nunique()
print(x1)

# x/all in training data
x1 = pert_df.loc[pert_df["pert"].isin(df4["pert_id"]), "pert"].nunique()
print(x1)

0
0


In [7]:
import pandas as pd
pert_df = pd.read_csv("/home/data/kaggle_data/myllia/pert_ids_all.csv")
print(sorted(pert_df["pert"].unique()))

['ACTB', 'ACTR3', 'ARID1A', 'ATP5F1A', 'ATP6V0C', 'BAX', 'BECN1', 'BIRC2', 'C1QBP', 'CASP3', 'CEBPB', 'CHD6', 'CHMP3', 'CLDN6', 'COX6C', 'CS', 'CSK', 'CTNNB1', 'CUL1', 'CUL2', 'DHX36', 'DLG5', 'DNMT1', 'DOT1L', 'DPF2', 'DPH2', 'EHMT1', 'EIF4B', 'ERBB3', 'FEN1', 'FLT4', 'FOSL1', 'FOSL2', 'FOXA1', 'FOXH1', 'FUBP1', 'GLUD1', 'GPI', 'GRB2', 'GSTZ1', 'HDAC2', 'HDAC3', 'HIRA', 'HK2', 'HMGCR', 'HMGN1', 'HMGXB4', 'HRAS', 'HSP90AA1', 'HSPA8', 'ITGAV', 'JUN', 'JUNB', 'KAT7', 'KDM1A', 'KDM2B', 'KDM3B', 'KDM6B', 'KIF20A', 'KLF10', 'KLF4', 'KMT2C', 'KMT2D', 'KMT2E', 'KRT18', 'MBTPS1', 'MDH2', 'MED1', 'MED12', 'MED13L', 'MED24', 'METTL17', 'MRE11', 'MTFR1', 'NDUFA7', 'NFATC2', 'NISCH', 'OXCT1', 'PBX1', 'PHF10', 'PHF14', 'PSMA1', 'PSME4', 'PTEN', 'RAB5A', 'RAF1', 'SETD1B', 'SETD2', 'SETD7', 'SETDB1', 'SIRT6', 'SLIRP', 'SMARCA2', 'SMARCB1', 'SMARCE1', 'SOX2', 'SP1', 'SRC', 'SRCAP', 'STAT3', 'STAT6', 'STRAP', 'SUPV3L1', 'TADA1', 'TAF13', 'TAFAZZIN', 'TCF3', 'TCF7L2', 'TP53', 'TRAM2', 'UBA1', 'UBE2D1', 

## VCC Data

In [10]:
basedir = "/home/data/single_cell/vcc_2025/"

In [11]:
adata2 = sc.read_h5ad(basedir + "adata_Training.h5ad")
print(adata2)
# print(adata.X)
print(adata2.obs)
print(adata2.var)
print(adata2.obs["target_gene"].nunique())

AnnData object with n_obs × n_vars = 221273 × 18080
    obs: 'target_gene', 'guide_id', 'batch'
    var: 'gene_id'
                                      target_gene  \
AAACAAGCAACCTTGTACTTTAGG-Flex_1_01          CHMP3   
AAACAAGCATTGCCGCACTTTAGG-Flex_1_01           AKT2   
AAACCAATCAATGTTCACTTTAGG-Flex_1_01          SHPRH   
AAACCAATCCCTCGCTACTTTAGG-Flex_1_01         TMSB4X   
AAACCAATCTAAATCCACTTTAGG-Flex_1_01          KLF10   
...                                           ...   
TTTGGACGTGGTGCAGATTCGGTT-Flex_3_16  non-targeting   
TTTGTGAGTAGTAGCAATTCGGTT-Flex_3_16          KDM1A   
TTTGTGAGTCCATCCTATTCGGTT-Flex_3_16  non-targeting   
TTTGTGAGTCCTGACAATTCGGTT-Flex_3_16          BIRC2   
TTTGTGAGTGGACACGATTCGGTT-Flex_3_16          EWSR1   

                                                                   guide_id  \
AAACAAGCAACCTTGTACTTTAGG-Flex_1_01                CHMP3_P1P2_A|CHMP3_P1P2_B   
AAACAAGCATTGCCGCACTTTAGG-Flex_1_01                  AKT2_P1P2_A|AKT2_P1P2_B   
AAACCAATCAA

In [12]:
df = adata2.to_df()
df.head()
print(df.columns.sort_values().to_list())

['A1CF', 'A2M', 'A2ML1', 'A3GALT2', 'A4GALT', 'A4GNT', 'AAAS', 'AACS', 'AADAC', 'AADACL2', 'AADACL3', 'AADACL4', 'AADAT', 'AAGAB', 'AAK1', 'AAMDC', 'AAMP', 'AANAT', 'AAR2', 'AARD', 'AARS', 'AARS2', 'AARSD1', 'AASDH', 'AASDHPPT', 'AASS', 'AATF', 'AATK', 'ABAT', 'ABCA1', 'ABCA10', 'ABCA12', 'ABCA13', 'ABCA2', 'ABCA3', 'ABCA4', 'ABCA5', 'ABCA6', 'ABCA7', 'ABCA8', 'ABCA9', 'ABCB1', 'ABCB10', 'ABCB11', 'ABCB4', 'ABCB5', 'ABCB6', 'ABCB7', 'ABCB8', 'ABCB9', 'ABCC1', 'ABCC11', 'ABCC12', 'ABCC2', 'ABCC3', 'ABCC4', 'ABCC5', 'ABCC8', 'ABCC9', 'ABCD1', 'ABCD2', 'ABCD3', 'ABCD4', 'ABCE1', 'ABCF1', 'ABCF2', 'ABCF3', 'ABCG1', 'ABCG2', 'ABCG4', 'ABCG5', 'ABCG8', 'ABHD1', 'ABHD10', 'ABHD11', 'ABHD12', 'ABHD12B', 'ABHD13', 'ABHD14A', 'ABHD14B', 'ABHD15', 'ABHD16A', 'ABHD16B', 'ABHD17B', 'ABHD17C', 'ABHD18', 'ABHD2', 'ABHD3', 'ABHD4', 'ABHD5', 'ABHD6', 'ABHD8', 'ABI1', 'ABI2', 'ABI3', 'ABI3BP', 'ABL1', 'ABL2', 'ABLIM1', 'ABLIM2', 'ABLIM3', 'ABO', 'ABR', 'ABRA', 'ABRACL', 'ABRAXAS1', 'ABRAXAS2', 'ABT1', '

In [13]:
perturbations = adata2.obs["target_gene"].unique().to_list()

# x/var in training data
x1 = pert_df.iloc[:60,:].loc[pert_df["pert"].isin(perturbations), "pert"].nunique()
print(x1)

# x/all in training data
x2 = pert_df.loc[pert_df["pert"].isin(perturbations), "pert"].nunique()
print(x2)

17
39


/tmp/ipykernel_5439/221966929.py:1: FutureWarning: Categorical.to_list is deprecated and will be removed in a future version. Use obj.tolist() instead
  perturbations = adata2.obs["target_gene"].unique().to_list()


## Replogle et al.

In [14]:
basedir3 = "/home/data/single_cell/Replogle_et_al/"

In [15]:
adata3 = sc.read_h5ad(basedir3 + "K562_essential_raw_singlecell_01.h5ad")
print(adata3)
print(adata3.obs)
print(adata3.var)
print(adata3.to_df().head())

AnnData object with n_obs × n_vars = 310385 × 8563
    obs: 'gem_group', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count'
    var: 'gene_name', 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano'
                     gem_group     gene          gene_id transcript  \
cell_barcode                                                          
AAACCCAAGAAATCCA-27         27     NAF1  ENSG00000145414       P1P2   
AAACCCAAGAACTTCC-31         31     BUB1  ENSG00000169679       P1P2   
AAACCCAAGAAGCCAC-34         34     UBL5  ENSG00000198258       P1P2   
AAACCCAAGAATAGTC-43         43  C9orf16  ENSG00000171159       P1P2   
AAACCCAAGACAGCGT-28         28    TIMM9  ENSG00000100575       P1P2   
...                        ...      ...              ...        ...   
TTTGTTGTCTGTCGTC-45         45  ATP6V1D  ENSG00000100554       P1P2   
TTTGTTGTCTGTCTCG-

## Jiang et al.

In [16]:
basedir4 = "/home/data/single_cell/Jiang_et_al/"

In [17]:
# adata4_1 = sc.read_h5ad(basedir4 + "TGFB_Perturb_seq.h5ad")
# print(adata4_1.obs)
# print(adata4_1.var)

In [18]:
# X = adata4_1.X[:5, :]
# if issparse(X):
#     X = X.toarray()

# print(pd.DataFrame(X, 
#     index=adata4_1.obs_names[:5], 
#     columns=adata4_1.var_names[:]))

In [19]:
# adata4_2 = sc.read_h5ad(basedir4 + "IFNB_Perturb_seq.h5ad")
# print(adata4_2.obs)
# print(adata4_2.var)

adata4_3 = sc.read_h5ad(basedir4 + "IFNG_Perturb_seq.h5ad")
print(adata4_3.obs)
print(adata4_3.var)

                            nCount_RNA  nFeature_RNA cell_type pathway  \
05_54_42_1_1_1_1_1_1_1_1_1     15381.0          5362      A549    IFNG   
08_58_81_1_1_1_1_1_1_1_1_1     13534.0          4995      A549    IFNG   
06_01_92_1_1_1_1_1_1_1_1_1     11520.0          4454      A549    IFNG   
05_33_45_1_1_1_1_1_1_1_1_1     11174.0          4575      A549    IFNG   
06_35_73_1_1_1_1_1_1_1_1_1     10807.0          3961      A549    IFNG   
...                                ...           ...       ...     ...   
07_01_13_1_1_1_1_1_1            4768.0          2717      A549    IFNG   
05_41_44_1_1_1_1_1_1            4725.0          2682      A549    IFNG   
07_55_80_1_1_1_1_1_1            4717.0          2668      A549    IFNG   
08_62_57_1_1_1_1_1_1            4700.0          2426      A549    IFNG   
06_28_94_1_1_1_1_1_1            4693.0          2482      A549    IFNG   

                            percent.mito sample_ID Batch_info     guide  \
05_54_42_1_1_1_1_1_1_1_1_1      5.96

## Total

In [ ]:
s0 = set(df2.columns)               # submission

s1 = set(adata.var["features"])     # myllia
s2 = set(df.columns)                # vcc
s3 = set(adata3.var["gene_name"])   # Replogle

overlap = s2 | s3
overlap2 = overlap & s0

print(len(s1))
print(len(s2))
print(f"Overlap: {len(overlap)}")

print(f"Containing {len(overlap2)}/5127 genes to predict")
# print(list(overlap))


19226
18080
Overlap: 8561
Containing 2419/5127 genes to predict


In [21]:
s0 = set(df2.columns)             # submission
s4_3 = set(adata4_3.var.index)    # Jiang_IFNG

overlap3 = s4_3 & s0

print(f"Containing {len(overlap3)}/5127 genes to predict")

Containing 4876/5127 genes to predict


In [22]:
df_s = pd.read_csv("/home/data/kaggle_data/myllia/pert_ids_all.csv")

In [ ]:
p0 = set(df_s["pert"].unique())
p0_1 = set(df_s.loc[:60, "pert"].unique())

p1 = set(adata.obs["sgrna_symbol"].unique())
p2 = set(adata2.obs["target_gene"].unique()) # vcc
p3 = set(adata3.obs["gene"].unique())       # replogle
p4_3 = set(adata4_3.obs["gene"].unique())

union = p1 | p2 | p3 | p4_3
union_2 = p2 | p3
pert_overlap = union & p0
pert_overlap1 = p1 & p0
pert_overlap2 = p2 & p0
pert_overlap3 = p3 & p0
pert_overlap4_3 = p4_3 & p0
pert_overlap_1 = union & p0_1
pert_overlap_2 = union_2 & p0

print(f"all: {len(pert_overlap)}/120")
print(f"p1: {len(pert_overlap1)}/120")
print(f"p2: {len(pert_overlap2)}/120")
print(f"p3: {len(pert_overlap3)}/120")
print(f"p4: {len(pert_overlap4_3)}/120")
print(f"val: {len(pert_overlap_1)}/120")
print(f"p2|p3 vs all: {len(pert_overlap_2)}/120")
print(p3)

all: 65/120
p1: 0/120
p2: 39/120
p3: 35/120
p4: 7/120
val: 30/120
p2|p3 vs all: 61/120
{'PRIM2', 'DHX33', 'GBF1', 'SUPT16H', 'SMG1', 'H2AFX', 'MARS', 'POTEI', 'PPP1R11', 'TRAPPC1', 'MRPL10', 'CEP85', 'CCAR1', 'PKMYT1', 'TTI1', 'RBM25', 'SART3', 'ETF1', 'NR2C2AP', 'NOP10', 'PSMC5', 'U2AF1', 'TIMM23B', 'LYRM4', 'CACNB3', 'PRPF31', 'GPN3', 'ORC1', 'FARS2', 'COG3', 'TFB2M', 'HNRNPC', 'MSRB1', 'PGPEP1', 'CNOT3', 'PNISR', 'VEZT', 'PSMB2', 'SRSF1', 'PAF1', 'TMX2', 'MBTPS1', 'RBM14', 'MVK', 'RANGAP1', 'SNIP1', 'TLCD1', 'XRCC5', 'DPH6', 'CDCA5', 'MED14', 'MED31', 'AHCTF1', 'TTF2', 'TUBGCP4', 'GTF2H4', 'CACTIN', 'GOLGA6L1', 'EXOSC4', 'SLC39A10', 'HMGA1', 'ORC3', 'TUBA1C', 'SYS1', 'RNF14', 'USP39', 'MRPS18A', 'WDR4', 'HSPA8', 'ZNF830', 'CD8B', 'RAD9A', 'RPL39', 'NUDT21', 'POLR2H', 'TERF1', 'CCDC115', 'NFATC2IP', 'MCMBP', 'BPTF', 'RBBP8', 'USP37', 'NUP205', 'TACC3', 'COA5', 'UBA3', 'RINT1', 'RPLP2', 'MKRN1', 'PSME2', 'CCNK', 'PSMB7', 'ISCU', 'ELOVL1', 'PSMG2', 'HIST1H2AB', 'FARSA', 'PSMB5', 'SAP13